## Setup

### Load Modules

In [ ]:
%load_ext autoreload
%autoreload 2

#General Import
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle
from os.path import join
from sklearn.model_selection import KFold
from matplotlib import colormaps as cmaps
from mne.filter import filter_data, resample
import scipy.stats as stats
import pandas as pd
from itertools import product
import xarray as xr
from scipy.signal import coherence, welch

#ML Import
from sklearn.decomposition import PCA, FastICA
from sklearn.preprocessing import StandardScaler, scale
import jPCA
import scipy.signal as signal
from statsmodels.tsa.stattools import grangercausalitytests
from scipy.signal import hilbert

#Electrophysiology Import
from spyeeg.models.TRF import TRFEstimator
from spyeeg.models.ERP import ERP_class
from spyeeg.utils import lag_matrix
import mne
import frites
from frites.simulations import sim_multi_suj_ephy
from frites.dataset import DatasetEphy
from frites.workflow import WfConnComod
from frites import set_mpl_style
import frites.conn as conn
from scipy.signal import welch
import spectral_connectivity 
from scipy.special import comb

#Performance Import
import time
import psutil

#Local Import
from stats_utils import cliffs_delta, cohen_d
from nice_utils import estimate_loop_time, decorator_loop
from preprocessing_utils import mono_to_bipolar, select_channels, available_regions, delete_channels, adj_scale, ica_shaft, pick_channels
from signal_utils import sparse_resample, lag_finder, sparse_realign
from viz_utils import create_matshow_gif, create_collection_gif

### Load Features

In [ ]:
# Acoustic Regressors
fs = 100
new_path = 'C:/Users/D-CAP/Documents/GitHub/witching-star/regressors/selected_regs.pkl'
new_data = pickle.load(open(new_path, 'rb'))
data_fs = new_data['fs']
new_regressors = new_data['regs']
new_names = new_data['regs_name']
ratio = fs/data_fs
new_duration = int(new_regressors.shape[0] * ratio) + 1

new_resamp = []
for i in range(new_regressors.shape[1]):
    name = new_names[i]
    if name in ['Intensity', 'Envelope Oganian', 'Envelope Derivative TF', 'F0 Loudness', 'SpectralFlux Filtered', 'SpectralFlux not_filtered']:
        new_reg = mne.filter.resample(new_regressors[:,i], up=100, down=data_fs)[:new_duration]
    elif name in ['peakEnv_tf', 'Syllabe Onset', 'p-syl', 'Phono']:
        new_reg = new_regressors[:,i] - np.min(new_regressors[:,i])
        new_reg = sparse_resample(new_reg, new_fs = fs, current_fs = data_fs)[:new_duration]
    else:
        print('wut')
    new_resamp.append(new_reg)
new_resamp = np.asarray(new_resamp).T
regressors = new_resamp
regressors_name = new_data['regs_name']


In [ ]:
# Renyi2 Array

path_renyi = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/renyi_array2.pickle"
renyi_data = pickle.load(open(path_renyi, 'rb'))
data_fs = renyi_data['fs']
X = np.roll(renyi_data['X'],4, axis=0)
onsets = np.where(X[:,5] >0)[0]
reg_renyi = np.zeros([regressors.shape[0], X.shape[1]])

for onset in onsets:
    new_onset = int(onset/data_fs*fs)
    for renyi_index in range(X.shape[1]):
        reg_renyi[new_onset, renyi_index] = X[onset,renyi_index]

regressors = np.hstack([regressors, reg_renyi])
regressors_name = regressors_name + renyi_data['names']
renyi_values_wrd = [float(renyi_name.split('renyi ')[1]) for renyi_name in renyi_data['names'][:20]]

### Load Neural Data

In [ ]:
# Load Broadband

data_subject = dict()
channels_subject = dict()
locations_subject = dict()
path_data = "D:/DataSEEG_Sorciere/BIDS/data_mne_fif"
for index_subject in range(1,40):
    try:
        name_subject = 'sub-{:03}'.format(index_subject)
        name_file = name_subject + '_task-iSpeech_speech-epo.fif'
        path_file = os.path.join(path_data, name_subject,'preprocessed','epochs',name_file)
        if os.path.isfile(path_file):
            mne_data = mne.read_epochs(path_file, verbose = False)
        elif os.path.isfile(os.path.join(path_data, name_subject,'preprocessed','monopolar',name_file)):
            path_file = os.path.join(path_data, name_subject,'preprocessed','monopolar',name_file)
            mne_data = mne.read_epochs(path_file, verbose = False)
        else:
            path_file = os.path.join(path_data, name_subject,'preprocessed','epochs','monopolar',name_file)
            mne_data = mne.read_epochs(path_file, verbose = False)
        mne_data_resample = mne_data.resample(fs, npad = 'auto', verbose = False)
        mne_data_resample.filter(0.3, 49, verbose = False)
        channels = mne_data_resample.ch_names
        montage = mne_data_resample.get_montage()
        ch_names = [n for n,_ in montage.get_positions()['ch_pos'].items()] 
        loc = (1e3 * np.stack([coord for _,coord in montage.get_positions()['ch_pos'].items()])).T  # store locations
        
        data_subject[index_subject] = mne_data_resample.get_data(copy = False)[0].T[:regressors.shape[0],:]
        channels_subject[index_subject] = channels
        locations_subject[index_subject] = loc
        print('Subject', index_subject, 'loaded')
    except:
        print('Error in subject', index_subject)

data_bipolar, channels_bipolar, locations_bipolar = mono_to_bipolar(data_subject, channels_subject, locations_subject)

In [ ]:
atlas_subject = dict()
path_data = "D:/DataSEEG_Sorciere/BIDS/data_mne_fif"
path_atlas = "D:/DataSEEG_Sorciere/BIDS/freesurfer"
for index_subject in range(1,40):
    try:
        name_subject = 'sub-{:03}'.format(index_subject)
        name_bipolar_atlas = 'elecbipolar2atlas.mat'
        name_monopolar_atlas = 'elec2atlas.mat'
        path_bipolar_atlas = os.path.join(path_atlas, name_subject,name_bipolar_atlas)
        path_monopolar_atlas = os.path.join(path_atlas, name_subject,name_monopolar_atlas)
        bipolar_atlas = get_bipolar_atlas(path_bipolar_atlas, atlas = 'Desikan_Killiany') #Destrieux
        atlas_subject[index_subject] = bipolar_atlas
    except:
        print('Atlas error in subject', index_subject)

subjects =  list(atlas_subject.keys())
new_atlas = dict()
for subject_index in atlas_subject:
    new_atlas[subject_index] = dict()
    for channels_bipolar in atlas_subject[subject_index]:
        new_name = channels_bipolar.split('-')[0] + '||' + channels_bipolar.split('-')[1]
        new_atlas[subject_index][new_name] = atlas_subject[subject_index][channels_bipolar][0]

# Pre-clustered FIT

In [ ]:
regressors_clusters = [233,258]
regressors_clusters = list(np.arange(233,253)) + [258]

clustering = 'NMF6'
montage_choice = 'bipo'

cluster_group = dict()
channels_subject_selection = dict()
for subject_index, subject_id in enumerate(data_bipolar):
    regressors_str = '_'.join(np.asarray(regressors_clusters).astype('str'))
    target_name = clustering + '_' + montage_choice + '_' +  str(subject_id) + '_' + regressors_str
    filename = 'MI_cluster/' + target_name + '.pickle'
    with open(filename, 'rb') as file:
        cluster_group[subject_id] = pickle.load(file)
    channels_subject_selection[subject_id] = []
    for cluster in cluster_group[subject_id]:
        channels_subject_selection[subject_id] += [i.replace('||','-') for i in cluster_group[subject_id][cluster]]
    channels_subject_selection[subject_id] 

In [51]:
montage = 'bipo' #bipo
montage_choice = 'bipo' #bipo
channel_selection = []
channel_selection_join = ''.join(channel_selection)
tmin = -2.5 #-0.5
tmax = 1.5 #1.3
step = 1
env_ref = 1
baseline_limits = [0,50]
moving_avg = 3
max_delay = 1.0
apply_baseline = False
baseline_str = (not apply_baseline) * 'no_baseline'
time_array = np.linspace(tmin, tmax,int((tmax-tmin)*fs))


for regressor_index in [258,233,253]: #[233,253,258]
    for subject_index in range(len(data_subject)):
        subject_id = list(data_subject.keys())[subject_index]
        eeg = data_bipolar[subject_id]
        channels = channels_bipolar[subject_id]
        eeg_mono = data_subject[subject_id]
        channels_mono = channels_subject[subject_id]
        #eeg_Hfa = data_Hfa_bipolar[subject_id]
        #channels_Hfa = channels_Hga_bipolar[subject_id]

        channel_selection = channels_subject_selection[subject_id]
        
        eeg_HT, channels_HT = pick_channels(eeg,channels, channel_select = channel_selection, exclude = False)
        #eeg_Hfa, channels_Hfa = pick_channels(eeg_Hfa, channels_Hfa, channel_select = channel_selection, exclude = False)
        print('\nSubject:',subject_index, '\nRegressor ID:', regressor_index)
        print("Computing MI over ", len(channels_HT), "channels")

        y1 = eeg_HT[:,:]
        #y_Hfa = eeg_Hfa[:,:]
        x1 = regressors[:y1.shape[0],[regressor_index]]


        signal = x1[:,0]

        erp = ERP_class(tmin = tmin, tmax = tmax, srate=100)

        if montage == 'mono':
            erp.add_events(y1_mono, signal, weight_events = False, record_weight = True)
            channels_name = channels_mono_HT
        elif montage == 'bipo':
            erp.add_events(y1, signal, weight_events = False, record_weight = True)
            channels_name = []
            for chan_name in channels_HT:
                channels_name.append(chan_name.replace('-', '||'))
        elif montage == 'ica':                
            y_ica, channels_ica = ica_shaft(y1_mono, channels_mono_HT, random_state = 0)
            erp.add_events(y_ica, signal, weight_events = False, record_weight = True)
            channels_name = channels_ica
        elif montage == 'Hfa':
            erp.add_events(y_Hfa, signal, weight_events = False, record_weight = True)
            channels_name = []
            for chan_name in channels_Hfa:
                channels_name.append(chan_name.replace('-', '||'))

        epoched_data = np.asarray(erp.evoked).transpose(0,2,1)
        if apply_baseline:
            baseline = np.repeat(epoched_data[:,:,baseline_limits].mean(-1), epoched_data.shape[-1]).reshape(epoched_data.shape)
            epoched_data = epoched_data - baseline
        epoched_reg = np.asarray(erp.weights)

        fit = conn.conn_fit(epoched_data, epoched_reg, roi=channels_name, times=time_array, mi_type='cc', gcrn=True,
            max_delay=max_delay, avg_delay=True, net=False, sfreq=fs,
            verbose=None)
        target_name = clustering + '_' + montage_choice + '_' +  str(subject_id) + '_' + regressors_str
        filename = 'FIT/FIT4_' + baseline_str + montage + '_avg_' + target_name + '_reg' + str(regressor_index) + '_sub' + str(subject_id) + '.nc'
        fit.to_netcdf(filename)
        print(subject_index, regressor_index )



Subject: 0 
Regressor ID: 258
Computing MI over  15 channels


Defining links (n_roi=15; directed=True; net=False, nb_min_links=None)
Compute FIT on 210 connectivity pairs (max_delay=1.0)
    Apply the Gaussian Copula Rank Normalization


0 258

Subject: 1 
Regressor ID: 258
Computing MI over  9 channels


Defining links (n_roi=9; directed=True; net=False, nb_min_links=None)
Compute FIT on 72 connectivity pairs (max_delay=1.0)
    Apply the Gaussian Copula Rank Normalization


1 258

Subject: 2 
Regressor ID: 258
Computing MI over  18 channels


...

30 253

Subject: 31 
Regressor ID: 253
Computing MI over  19 channels


Defining links (n_roi=19; directed=True; net=False, nb_min_links=None)
Compute FIT on 342 connectivity pairs (max_delay=1.0)
    Apply the Gaussian Copula Rank Normalization
Defining links (n_roi=6; directed=True; net=False, nb_min_links=None)
Compute FIT on 30 connectivity pairs (max_delay=1.0)
    Apply the Gaussian Copula Rank Normalization


31 253

Subject: 32 
Regressor ID: 253
Computing MI over  6 channels


Defining links (n_roi=13; directed=True; net=False, nb_min_links=None)
Compute FIT on 156 connectivity pairs (max_delay=1.0)
    Apply the Gaussian Copula Rank Normalization


32 253

Subject: 33 
Regressor ID: 253
Computing MI over  13 channels
33 253
